# 45 — Experimental hybrid-earthquake screens

Test two hypotheses on the selected event data windows: **HF envelope peak preceding LF envelope peak by 1–10 s**, and a separate sustained high-to-low instantaneous-frequency transition. These are candidate screens, not validated hybrid labels. Ordinary phase/coda differences, mixed events, filtering and changing signal-to-noise can produce similar patterns.

The production metric engine already records `feature.hf.peak_time_s`, `feature.lf.peak_time_s` and `feature.hf_lf.peak_delay_s` (LF minus HF). This notebook reuses those definitions and adds diagnostic thresholds, plots, controls and optional labelled evaluation. By default it investigates only the first **30 seconds** of the event window; set `event_search_seconds=null` to use the whole selected event window.

All outputs go to `results_root/testing/hybrid/<test-id>/`. Production window decisions, cached event metrics and classifications are not changed. Read settings from **hybrid_test_config.json**. A small spread-out subset is the default; the main archive selection stays unchanged.

In [ ]:
from pathlib import Path
import sys, json
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in _candidates if (p / 'project_config.json').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start inside the Pinatubo repository.')
for p in [PROJECT_ROOT, PROJECT_ROOT / '20_analysis']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
from pinatubo_project import runtime_environment, test_directory
runtime_environment()
from IPython.display import display, Image, Markdown
from hybrid_diagnostics import run_hybrid_lab, synthetic_controls
ANALYSIS_DIR = PROJECT_ROOT / '20_analysis'
settings = json.loads((ANALYSIS_DIR / 'hybrid_test_config.json').read_text())
# Optional in-memory overrides: shared production config is not rewritten.
OVERRIDES = {}  # e.g. {'max_files': 25, 'minimum_peak_band_snr': 4.0}
settings.update(OVERRIDES)
display(settings)


## Synthetic controls before real-data interpretation

Use a designed HF→LF pair, reversed order, synchronous peaks, HF-only, LF-only and noise-only signals. Controls verify the sign convention and expose false-positive mechanisms; success here is not validation on real volcano signals. The peak method uses smoothed analytic envelopes in zero-phase LF/HF bands. Both peaks must exceed a configurable matched-PSD band-SNR threshold. No arbitrary amplitude/count threshold is shared across stations.

In [ ]:
CONTROL_DIR = test_directory('hybrid', 'synthetic_controls')
controls = synthetic_controls(settings, CONTROL_DIR)
display(controls[['control', 'lf_minus_hf_peak_delay_s', 'lf_peak_band_snr', 'hf_peak_band_snr',
                  'peak_delay_candidate', 'if_candidate', 'screen_class', 'flags']])
for name in ['hf_then_lf', 'lf_then_hf', 'synchronous']:
    display(Image(filename=str(CONTROL_DIR / f'{name}.png')))


## Real event-window examples

Use the policy chosen in notebook 30 (fixed five-second or detected noise/event windows). The test reads a bounded subset, computes its own diagnostic features and records exact source identities/parameters. Peak times are relative to the diagnostic event-window start; positive lag means LF later.

The separate IF screen applies the configured common bandpass, uses signed Hilbert phase derivatives, masks low-envelope/edge estimates and checks valid coverage, negative-frequency fraction and envelope/noise ratio. It requires sustained high and low states separated by the configured interval and minimum frequency drop. Its state-transition delay is not the same quantity as the LF/HF peak delay. Negative estimates remain in saved summaries; windows dominated by them are excluded from the candidate decision.

In [ ]:
OUT, scores, events, errors = run_hybrid_lab(ANALYSIS_DIR, overrides=OVERRIDES)
print('Test outputs:', OUT)
print('Traces:', len(scores), '; recording files:', len(events), '; errors:', len(errors))
display(scores.head(40))
display(events)
display(errors)
for path in sorted((OUT / 'figures').glob('*.png')):
    display(Image(filename=str(path)))


## Labelled evaluation and threshold sensitivity

Fill `labels_template.csv` with explicit human `is_hybrid` labels and reasons, then set `labels_csv` in the test config or use the call below. Blank labels remain unknown. Evaluation reports unresolved coverage separately; undefined precision/recall remain missing. Hold out entire recording events (not individual components) when selecting thresholds, and include ordinary earthquakes, LP signals, noise, clipped/weak traces and overlapping events.

The experimental event-level table counts **distinct stations**, not components. Requiring agreement can reduce isolated artifacts but is not proof of a common source mechanism. Sensor/gain history and timing uncertainty remain relevant. No label is written to the production EnhancedCatalog.

In [ ]:
from hybrid_diagnostics import evaluate_labels
LABELS_CSV = settings.get('labels_csv')
if LABELS_CSV:
    evaluation = evaluate_labels(scores, Path(LABELS_CSV).expanduser())
    evaluation.to_csv(OUT / 'label_evaluation.csv', index=False)
    display(evaluation)
else:
    print('No real-data classification accuracy claimed: no human labels supplied.')
    print('Label template:', OUT / 'labels_template.csv')
# For sensitivity: rerun with OVERRIDES changing delay bounds, bands, SNR,
# IF drop/coverage/gating, keeping the same labelled event-level holdout.
